# Part 1 : Snowflake 基礎

Part1 では、普段の案件では触れる機会が少ないかもしれない **Snowflake ならではの基本機能**を一通り体験します。
この後の AI パートで使う SNS 投稿データ(`sns_mentions` テーブル)を題材に進めます。

| # | 機能 | ポイント |
|---|---|---|
| 1 | アダプティブウェアハウス | サイズ指定なしでワークロードに応じて自動スケール |
| 2 | ゼロコピークローン | ストレージを消費せずに瞬時にテーブルを複製 |
| 3 | データ修正と SWAP | 開発用コピーで修正し、本番とアトミックに入れ替え |
| 4 | UNDROP | 誤って削除したテーブルを瞬時に復旧 |
| 5 | 動的テーブル | SELECT 文を書くだけで自動更新されるパイプライン |

In [ ]:
%%sql -r dataframe_0_1
-- コンテキスト設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE AI_HANDSON_WH;
USE DATABASE AI_HANDSON_DB;
USE SCHEMA ANALYTICS;

---
## 1. ウェアハウスの作成

この後のハンズオンで利用するウェアハウスとして、アダプティブウェアハウスを作成します。
この後の手順ではSQLで作成する流れになっていますが、GUIでも作成可能です。

合わせて確認してみてください。

> **GUI での作成方法**
> 
> Snowsight 左メニュー **コンピュート → ウェアハウス → + ウェアハウス** で、タイプ に **適応型** を選択します。
> ※ACCOUNT_ADMINなどウェアハウス作成権限があるロールを選択して行ってください

### SQL で作成する

In [ ]:
%%sql -r dataframe_0_2
-- アダプティブウェアハウスを作成
CREATE OR REPLACE ADAPTIVE WAREHOUSE AI_HANDSON_ADAPTIVE_WH
    MAX_QUERY_PERFORMANCE_LEVEL = MEDIUM
    QUERY_THROUGHPUT_MULTIPLIER = 2
    COMMENT = 'Part 1 アダプティブウェアハウス ハンズオン用';

In [ ]:
%%sql -r dataframe_0_3
-- 作成結果を確認
-- 確認ポイント:
--   1. "type" = ADAPTIVE → 従来の STANDARD でなくアダプティブウェアハウスとして作成されている
--   2. "size" = NULL → アダプティブWHにはサイズ（XS/S/M/...）の概念がなく、ワークロードに応じて自動スケール
--   3. "state" = ENABLED → アダプティブWHは SUSPEND/RESUME ではなく ENABLE/DISABLE で制御する（STARTED/SUSPENDED にはならない）
--   4. "max_query_performance_level" = MEDIUM → 1クエリあたりに使えるコンピュートの上限（XSMALL〜X4LARGE）
--   5. "query_throughput_multiplier" = 2 → 同時実行性能の倍率（デフォルト2、大きいほど並列処理に強い。0=無制限）
SHOW WAREHOUSES LIKE 'AI_HANDSON_ADAPTIVE_WH';

In [ ]:
%%sql -r dataframe_0_4
-- アダプティブウェアハウスでクエリを実行してみる
USE WAREHOUSE AI_HANDSON_ADAPTIVE_WH;

-- 商品別の売上を集計
SELECT
    p.product_name,
    p.category,
    SUM(s.units_sold)   AS total_units,
    SUM(s.sales_amount) AS total_sales
FROM daily_sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_name, p.category
ORDER BY total_sales DESC;

---
## 2. ゼロコピークローン

本番テーブルを直接いじるのは怖いので、まず**開発用のコピー**を作ります。

Snowflake の `CLONE` は**ゼロコピークローン**と呼ばれ、データを物理的にコピーしません。
メタデータだけを複製するので、テーブルのサイズに関係なく**一瞬で完了し、追加のストレージも消費しません**。
クローン後に変更した部分だけが、新たにストレージとして記録されます。

In [ ]:
%%sql -r dataframe_0_5
-- 本番テーブルを確認(platform 別の件数)
SELECT platform, COUNT(*) AS post_count
FROM sns_mentions
GROUP BY platform
ORDER BY platform;

In [ ]:
%%sql -r dataframe_0_6
-- 開発用のクローンを作成
CREATE OR REPLACE TABLE sns_mentions_dev CLONE sns_mentions;

In [ ]:
%%sql -r dataframe_0_7
-- クローンの中身を確認(本番と同じデータが入っている)
SELECT post_id, platform, username, content, likes, hashtags
FROM sns_mentions_dev
ORDER BY post_id
LIMIT 10;

---
## 3. データ修正と SWAP

`platform` 列には `twitter` という値が入っていますが、サービス名は現在 **X** に変わっています。
開発用のクローンで値を修正し、問題なければ本番テーブルと入れ替えます。

In [ ]:
%%sql -r dataframe_0_8
-- 開発用テーブルで twitter を x に修正
UPDATE sns_mentions_dev
SET platform = 'x'
WHERE platform = 'twitter';

In [ ]:
%%sql -r dataframe_0_9
-- 修正結果を確認(本番はまだ twitter のまま)
SELECT 'dev'  AS env, platform, COUNT(*) AS post_count FROM sns_mentions_dev GROUP BY platform
UNION ALL
SELECT 'prod' AS env, platform, COUNT(*) AS post_count FROM sns_mentions     GROUP BY platform
ORDER BY env, platform;

修正が問題なさそうなので、本番テーブルと入れ替えます。

`SWAP WITH` は 2 つのテーブルの**メタデータとデータをアトミックに交換**します。
アプリケーションから見ると、`sns_mentions` が一瞬で修正後のデータに切り替わります。

In [ ]:
%%sql -r dataframe_0_10
-- 本番と開発を入れ替える
ALTER TABLE sns_mentions SWAP WITH sns_mentions_dev;

In [ ]:
%%sql -r dataframe_0_11
-- 本番が修正後のデータになっていることを確認
SELECT platform, COUNT(*) AS post_count
FROM sns_mentions
GROUP BY platform
ORDER BY platform;

---
## 4. UNDROP によるデータ復旧

後片付けで開発用テーブルを消すつもりが……**本番テーブルを誤って削除**してしまいました。

In [ ]:
%%sql -r dataframe_0_12
-- 誤って本番テーブルを削除
DROP TABLE sns_mentions;

In [ ]:
%%sql -r dataframe_0_13
-- テーブルが存在しないためエラーになる
SELECT COUNT(*) FROM sns_mentions;

Snowflake には **Time Travel** という仕組みがあり、削除されたテーブルも保持期間内(デフォルト 1 日)であれば `UNDROP` で復旧できます。
バックアップからのリストアは不要です。

In [ ]:
%%sql -r dataframe_0_14
-- テーブルを復旧
UNDROP TABLE sns_mentions;

In [ ]:
%%sql -r dataframe_0_15
-- 修正後のデータのまま戻っていることを確認
SELECT platform, COUNT(*) AS post_count
FROM sns_mentions
GROUP BY platform
ORDER BY platform;

In [ ]:
%%sql -r dataframe_0_16
-- 今度こそ開発用テーブルを削除
DROP TABLE sns_mentions_dev;

---
## 5. 動的テーブルによるパイプライン

**動的テーブル(Dynamic Table)** は、「欲しい結果」を SELECT 文で定義するだけで、
元テーブルの変更を検知して**自動で更新**してくれるテーブルです。

- `TARGET_LAG` : 元データからどのくらいの遅れまで許容するか(最小 1 分)
- `WAREHOUSE` : 更新処理に使うウェアハウス

Stream や Task を組み合わせて書いていた ETL を、SQL 1 本で宣言的に置き換えられます。

ここでは次の 2 段のパイプラインを作ります。

```
sns_mentions ──▶ DT_SNS_HASHTAGS ──▶ DT_HASHTAG_SUMMARY
 (元データ)       (ハッシュタグを展開)   (ハッシュタグ別に集計)
```

### Step 1 : ハッシュタグを 1 行ずつに展開する

`hashtags` 列は ARRAY 型で、1 投稿に複数のハッシュタグが入っています。
`FLATTEN` を使うと、配列の要素ごとに行を生成できます。

In [ ]:
%%sql -r dataframe_0_17
-- 元データの hashtags 列を確認
SELECT post_id, platform, hashtags
FROM sns_mentions
LIMIT 5;

In [ ]:
%%sql -r dataframe_0_18
-- 1段目 : ハッシュタグを展開する動的テーブル
CREATE OR REPLACE DYNAMIC TABLE DT_SNS_HASHTAGS
    TARGET_LAG = '1 minute'
    WAREHOUSE  = AI_HANDSON_WH
AS
SELECT
    m.post_id,
    m.platform,
    m.posted_at,
    m.likes,
    h.value::VARCHAR AS hashtag
FROM sns_mentions m,
    LATERAL FLATTEN(input => m.hashtags) h;

In [ ]:
%%sql -r dataframe_0_19
SELECT * FROM DT_SNS_HASHTAGS LIMIT 20;

### Step 2 : ハッシュタグ別に集計する

1 段目の動的テーブルを元に、さらに動的テーブルを作ります。
このように**動的テーブル同士をつなげてパイプラインを組める**のが特徴です。

In [ ]:
%%sql -r dataframe_0_20
-- 2段目 : ハッシュタグ × platform 別に集計する動的テーブル
CREATE OR REPLACE DYNAMIC TABLE DT_HASHTAG_SUMMARY
    TARGET_LAG = '1 minute'
    WAREHOUSE  = AI_HANDSON_WH
AS
SELECT
    hashtag,
    platform,
    COUNT(*)              AS post_count,
    ROUND(AVG(likes), 1)  AS avg_likes
FROM DT_SNS_HASHTAGS
GROUP BY hashtag, platform;

In [ ]:
%%sql -r dataframe_0_21
SELECT * FROM DT_HASHTAG_SUMMARY
ORDER BY post_count DESC
LIMIT 20;

### Step 3 : 元データを追加して自動更新を確認する

新しい投稿を 1 件追加します。
動的テーブルを直接更新しなくても、`TARGET_LAG`(1 分)以内に 2 段とも自動で反映されます。

In [ ]:
%%sql -r dataframe_0_22
-- 新しい投稿を 1 件追加(新しいハッシュタグ「truestarハンズオン」付き)
INSERT INTO sns_mentions (post_id, platform, post_type, username, display_name,
                          content, posted_at, likes, retweets, replies, hashtags)
SELECT
    'SNS-X-2026-99999', 'x', 'post', '@truestar_user', 'truestar ハンズオン参加者',
    'Snowflake のハンズオンで動的テーブルを試しています!',
    CURRENT_TIMESTAMP(), 100, 10, 5,
    ARRAY_CONSTRUCT('truestarハンズオン', 'グレイシャースタイル');

1 分ほど待ってから次のセルを実行してください。
待っている間に、下の **DAG(依存関係グラフ)** を確認してみましょう。

> 結果が 0 件の場合は、まだ動的テーブルが更新されていません。少し待って再実行してください。

In [ ]:
%%sql -r dataframe_0_23
-- 新しいハッシュタグが集計に反映されていることを確認
SELECT *
FROM DT_HASHTAG_SUMMARY
WHERE hashtag = 'truestarハンズオン';

### DAG(有向非循環グラフ)でパイプラインを可視化する

動的テーブル同士の依存関係は、Snowsight 上でグラフとして確認できます。

1. Snowsight 左メニュー **カタログ → データベースエクスプローラー** を開く
2. **AI_HANDSON_DB → ANALYTICS → 動的テーブル → DT_HASHTAG_SUMMARY** を選択
3. **グラフ** タブを開く

`SNS_MENTIONS → DT_SNS_HASHTAGS → DT_HASHTAG_SUMMARY` の流れと、各テーブルの更新状況(最終更新時刻、ラグ)を確認できます。
**更新履歴** タブでは、各リフレッシュが増分(INCREMENTAL)で実行されたかも確認できます。

In [ ]:
%%sql -r dataframe_0_24
-- SQL で更新状況を確認することもできる
SHOW DYNAMIC TABLES IN SCHEMA AI_HANDSON_DB.ANALYTICS;

---
## まとめ

- **アダプティブウェアハウス** : サイズを意識せずに使える
- **ゼロコピークローン + SWAP** : 本番を止めずに、安全に修正して切り替えられる
- **UNDROP** : 誤削除もすぐに戻せる
- **動的テーブル** : SELECT を書くだけで、自動更新されるパイプラインを作れる

次の Part2 では、同じ `sns_mentions` テーブルに **Cortex AI 関数**を使っていきます。